# Loader

In [2]:
from pathlib import Path
from typing import List
from src.models.document import Document, DocumentMetadata

def load_texts(directory_path: Path) -> List[Document]:
    """
    Load all plain text files from a directory and establish metadata foundations.
    """
    directory_path = Path(directory_path)

    if not directory_path.exists():
        raise FileNotFoundError(f"Directory not found: {directory_path}")

    text_files = sorted(directory_path.glob("*.txt"))
    documents: List[Document] = []

    for txt_file in text_files:
        content = txt_file.read_text(encoding="utf-8")

        metadata = DocumentMetadata(
            source=txt_file.name,
            file_type="txt",
            page=None,
            section=None
        )

        documents.append(
            Document(
                page_content=content,
                metadata=metadata
            )
        )
        
    return documents

In [3]:
text_docs = load_texts(directory_path=("../documents/text"))

In [4]:
text_docs

[Document(page_content='Apollo 11 (July 16–24, 1969) was the American spaceflight that first landed humans on the Moon, and the fifth crewed mission of NASA\'s Apollo program. The mission was crewed by Commander Neil Armstrong, Command Module Pilot Michael Collins, and Lunar Module Pilot Edwin "Buzz" Aldrin, all of whom were on their second and final spaceflight.\n\nLaunched atop a Saturn V rocket from Kennedy Space Center in Florida on July 16 at 13:32 UTC, the Apollo spacecraft consisted of three parts: the command module (CM), which housed the three astronauts and was the only part to return to Earth; the service module (SM), which provided propulsion, electrical power, oxygen, and water to the command module; and the Lunar Module (LM), which had two stages—a descent stage with a large engine and fuel tanks for landing on the Moon, and a lighter ascent stage containing a cabin for two astronauts and a small engine to return them to lunar orbit. After a three-day transit, Armstrong a

In [10]:
with open("loaded_text_docs.txt", "w", encoding="utf-8") as f:
    for doc in text_docs: 
        f.write(f"{doc}\n")

# Cleaner

In [5]:
import re
from typing import List
from src.models.document import Document

class TextCleaner:
    def __init__(self):
        # 1. Removes Wikipedia style bracketed footnotes e.g., [1], [12][13], [22]
        self.CITATION_PATTERN = re.compile(r"\[\d+\]")
        
        # 2. Catches weird formatting typography artifacts (hair spaces, special dashes)
        self.EM_DASH_PATTERN = re.compile(r"[\u2014\u2015]")
        self.HAIR_SPACE_PATTERN = re.compile(r"[\u200a\u200b]")
        
        # 3. Targets multiple periods/ellipses used loosely in text transcriptions
        self.ELLIPSIS_PATTERN = re.compile(r"\s*\.{3,}\s*")
        
        # 4. Matches exactly 3 or more consecutive newlines to squeeze them down
        self.EXCESS_NEWLINES_PATTERN = re.compile(r"\n{3,}")

    def repair_paragraph_flow(self, text: str) -> str:
        """
        Stitches back single newlines that fragment standard sentences 
        while preserving true double-newline paragraph blocks.
        """
        # Split by explicit structural paragraph breaks
        paragraphs = text.split("\n\n")
        cleaned_paragraphs = []
        
        for para in paragraphs:
            # Replace single internal newlines within a single paragraph with a normal space
            joined_para = para.replace("\n", " ")
            # Collapse multiple spaces down to one
            joined_para = re.sub(r"\s+", " ", joined_para)
            cleaned_paragraphs.append(joined_para.strip())
            
        return "\n\n".join(cleaned_paragraphs)

    def clean_text(self, text: str) -> str:
        # Step 1: Strip noisy inline text artifacts
        text = self.CITATION_PATTERN.sub("", text)
        text = self.EM_DASH_PATTERN.sub(" - ", text)
        text = self.HAIR_SPACE_PATTERN.sub(" ", text)
        text = self.ELLIPSIS_PATTERN.sub(" ... ", text)
        
        # Step 2: Fix newline fragmentation and flow structure
        text = self.repair_paragraph_flow(text)
        text = self.EXCESS_NEWLINES_PATTERN.sub("\n\n", text)
        
        return text.strip()

    def clean_documents(self, documents: List[Document]) -> List[Document]:
        cleaned_documents = []
        for document in documents:
            cleaned_content = self.clean_text(document.page_content)
            cleaned_documents.append(
                Document(
                    page_content=cleaned_content,
                    metadata=document.metadata
                )
            )
        return cleaned_documents

In [6]:
cleaner = TextCleaner()
cleaned_txt_docs = cleaner.clean_documents(text_docs)

In [8]:
cleaned_txt_docs

[Document(page_content='Apollo 11 (July 16–24, 1969) was the American spaceflight that first landed humans on the Moon, and the fifth crewed mission of NASA\'s Apollo program. The mission was crewed by Commander Neil Armstrong, Command Module Pilot Michael Collins, and Lunar Module Pilot Edwin "Buzz" Aldrin, all of whom were on their second and final spaceflight.\n\nLaunched atop a Saturn V rocket from Kennedy Space Center in Florida on July 16 at 13:32 UTC, the Apollo spacecraft consisted of three parts: the command module (CM), which housed the three astronauts and was the only part to return to Earth; the service module (SM), which provided propulsion, electrical power, oxygen, and water to the command module; and the Lunar Module (LM), which had two stages - a descent stage with a large engine and fuel tanks for landing on the Moon, and a lighter ascent stage containing a cabin for two astronauts and a small engine to return them to lunar orbit. After a three-day transit, Armstrong

In [9]:
with open("cleaned_text_docs.txt", "w", encoding="utf-8") as f:
    for doc in cleaned_txt_docs: 
        f.write(f"{doc}\n")

# Chunker

In [11]:
from pathlib import Path
from typing import List

from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    MarkdownHeaderTextSplitter,
)
from src.models.document import Document, DocumentMetadata

# =========================================================
# SPLITTER CONFIGURATION
# =========================================================

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, chunk_overlap=100, separators=["\n\n", "\n", ". ", " ", ""]
)

markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[
        ("#", "header_1"),
        ("##", "header_2"),
        ("###", "header_3"),
    ],
    strip_headers=False,  # Crucial for preserving structural layout context
)

# =========================================================
# HELPER FUNCTIONS
# =========================================================


def extract_section_name(metadata: dict) -> str | None:
    """Extract the most specific available markdown header."""
    return (
        metadata.get("header_3") or metadata.get("header_2") or metadata.get("header_1")
    )


def generate_chunk_id(source: str, chunk_index: int, page: int | None = None) -> str:
    """Generate deterministic, globally unique readable chunk IDs."""
    stem = Path(source).stem.lower().replace(" ", "_")
    if page is not None:
        return f"{stem}_page_{page}_chunk_{chunk_index}"
    return f"{stem}_chunk_{chunk_index}"


def create_chunk_document(
    content: str,
    original_metadata: DocumentMetadata,
    chunk_index: int,
    section: str | None = None,
) -> Document:
    """Create a chunked Document object with enriched metadata."""
    chunk_id = generate_chunk_id(
        source=original_metadata.source,
        page=original_metadata.page,
        chunk_index=chunk_index,
    )

    parent_document_id = Path(original_metadata.source).stem.lower().replace(" ", "_")

    metadata = DocumentMetadata(
        source=original_metadata.source,
        file_type=original_metadata.file_type,
        page=original_metadata.page,
        section=section,
        chunk_id=chunk_id,
        chunk_index=chunk_index,
        parent_document_id=parent_document_id,
    )

    return Document(page_content=content, metadata=metadata)


# =========================================================
# FIXED MAIN DISPATCHER & STRUCTURAL LOOPS
# =========================================================


def chunk_documents_new(documents: List[Document]) -> List[Document]:
    """
    Main chunking dispatcher with size constraints enforced across all document routes.
    """
    all_chunks = []
    document_counters = {}

    for document in documents:
        file_type = document.metadata.file_type.lower()
        source_file = document.metadata.source

        if source_file not in document_counters:
            document_counters[source_file] = 0

        # --- PROCESS PDF AND MD DOCUMENTS (Hybrid Strategy) ---
        if file_type in ["pdf", "md", "markdown"]:
            header_splits = markdown_splitter.split_text(document.page_content)
            last_active_section = None

            for split in header_splits:
                extracted_section = extract_section_name(split.metadata)

                if extracted_section:
                    last_active_section = extracted_section

                # Enforce chunk_size constraints via Recursive Character Splitting
                recursive_chunks = recursive_splitter.split_text(split.page_content)

                for chunk_text in recursive_chunks:
                    chunk_doc = create_chunk_document(
                        content=chunk_text,
                        original_metadata=document.metadata,
                        chunk_index=document_counters[source_file],
                        section=last_active_section,
                    )
                    all_chunks.append(chunk_doc)
                    document_counters[source_file] += 1

        # --- PROCESS PLAIN TEXT CHUNKS ---
        elif file_type == "txt":
            chunks = recursive_splitter.split_text(document.page_content)

            for chunk_text in chunks:
                chunk_doc = create_chunk_document(
                    content=chunk_text,
                    original_metadata=document.metadata,
                    chunk_index=document_counters[source_file],
                    section=None,
                )
                all_chunks.append(chunk_doc)
                document_counters[source_file] += 1

        else:
            raise ValueError(f"Unsupported file type: {file_type}")

    return all_chunks


w:\Data-Science\Projects\8.FullStack-RAG-Application-Project-Atman\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
text_chunks = chunk_documents_new(cleaned_txt_docs)

In [13]:
text_chunks?

Type:        list
String form: [Document(page_content='Apollo 11 (July 16–24, 1969) was the American spaceflight that first land <...> e, section=None, chunk_id='voyager_1_chunk_12', chunk_index=12, parent_document_id='voyager_1'))]
Length:      85
Docstring:  
Built-in mutable sequence.

If no argument is given, the constructor creates a new empty list.
The argument must be an iterable if specified.

In [14]:
text_chunks

[Document(page_content='Apollo 11 (July 16–24, 1969) was the American spaceflight that first landed humans on the Moon, and the fifth crewed mission of NASA\'s Apollo program. The mission was crewed by Commander Neil Armstrong, Command Module Pilot Michael Collins, and Lunar Module Pilot Edwin "Buzz" Aldrin, all of whom were on their second and final spaceflight.', metadata=DocumentMetadata(source='Apollo 11.txt', file_type='txt', page=None, section=None, chunk_id='apollo_11_chunk_0', chunk_index=0, parent_document_id='apollo_11')),
 Document(page_content='Launched atop a Saturn V rocket from Kennedy Space Center in Florida on July 16 at 13:32 UTC, the Apollo spacecraft consisted of three parts: the command module (CM), which housed the three astronauts and was the only part to return to Earth; the service module (SM), which provided propulsion, electrical power, oxygen, and water to the command module; and the Lunar Module (LM), which had two stages - a descent stage with a large engi

In [15]:
with open("text_chunks.txt", "w", encoding="utf-8") as f:
    for chunk in text_chunks: 
        f.write(f"{chunk}\n")